# 🛠️ Aula 14 — Introdução à Automação de GPUs com Bash

**Curso:** Introdução a Arquitetura de Computadores — Senac  
**Bloco 3:** Automação — da programação paralela à operação contínua e supervisionada  

---

## 🎯 Objetivo da Aula
Automatizar o **monitoramento de GPUs** com scripts **Bash** usando `nvidia-smi`, agendar a coleta com **cron/systemd**, gerar **dashboards com gnuplot/Matplotlib** e integrar com o **Google Sheets** — garantindo operação contínua e eficiente em ambientes de IA.

## 📌 Situação de Aprendizagem
O servidor de treinamento travou durante a madrugada: a GPU atingiu **95 °C** e o job falhou silenciosamente. Ninguém percebeu até a manhã seguinte. Precisamos de um sistema de monitoramento **24h/7d** que colete métricas a cada **5 s**, salve em **CSV**, gere **alertas de temperatura** e publique um dashboard diário no **Google Sheets** — tudo via scripts Bash agendados com cron.

### ⚙️ PASSO CRÍTICO: Ativar a GPU no Google Colab
1. Menu superior **Ambiente de execução** (*Runtime*) ➔ **Alterar tipo de ambiente de execução** (*Change runtime type*).
2. Em **Acelerador de hardware**, escolha **T4 GPU**.
3. Clique em **Salvar**.

> 💡 **Sem GPU?** O notebook detecta a ausência do `nvidia-smi` e entra em **modo simulado**, permitindo acompanhar toda a aula mesmo no runtime apenas com CPU.

---

## 🔄 Pipeline de Automação

```
nvidia-smi --query-gpu=...   →   monitor_gpu.sh   →   gpu_log.csv
     (coleta métricas)            (loop Bash)          (armazenamento local)
                                                              │
                                                              ▼
                   gnuplot / Google Sheets   ←   cron / systemd
                          (visualização)            (agendamento automático)
```


## 1. Verificação do Ambiente

Antes de escrever scripts, precisamos saber **onde estamos rodando**: existe GPU NVIDIA? O `nvidia-smi` está disponível?


In [ ]:
# @title 🔍 Detectar GPU e nvidia-smi no ambiente
# ============================================================================
# OBJETIVO: Descobrir se o ambiente possui GPU NVIDIA e o utilitário nvidia-smi.
# O resultado define se usaremos dados REAIS ou SIMULADOS nas próximas células.
# ============================================================================
import shutil  # shutil.which() localiza um executável no PATH
import subprocess  # subprocess executa comandos do sistema

# shutil.which('nvidia-smi') devolve o caminho do executável ou None se não existir
CAMINHO_NVIDIA_SMI = shutil.which("nvidia-smi")
TEM_GPU = CAMINHO_NVIDIA_SMI is not None  # True = monitoramento real

if TEM_GPU:
    print(f"✅ nvidia-smi encontrado em: {CAMINHO_NVIDIA_SMI}")
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout)
else:
    print("⚠️  nvidia-smi NÃO encontrado — a aula seguirá em MODO SIMULADO (dados sintéticos).")
    print("   Para dados reais: Runtime ➔ Change runtime type ➔ T4 GPU.")


## 2. Teoria: `nvidia-smi` — Opções e Queries Úteis

`nvidia-smi` é a **ferramenta CLI oficial da NVIDIA**. A opção `--query-gpu` extrai métricas **estruturadas** (ideais para scripts), com `--format=csv` controlando a saída.

| Grupo | Métricas |
| :--- | :--- |
| **Temperatura e energia** | `temperature.gpu`, `temperature.memory`, `power.draw`, `power.limit` |
| **Utilização e memória** | `utilization.gpu`, `utilization.memory`, `memory.used`, `memory.total` |
| **Clocks** | `clocks.current.graphics`, `clocks.current.memory`, `fan.speed` |

> ⚠️ No **T4 do Colab** alguns campos são `N/A` (ex.: `fan.speed` e, às vezes, `power.limit`). Os scripts tratam isso naturalmente, pois salvam o texto `N/A`.


In [ ]:
# @title 🧪 Referência rápida de queries do nvidia-smi (executa se houver GPU)
# ============================================================================
# OBJETIVO: Demonstrar as principais formas de consulta do nvidia-smi.
# ============================================================================
if TEM_GPU:
    # ── Resumo completo (visão humana) ──────────────────────────────────────
    !nvidia-smi

    print("\n" + "="*70)
    print("QUERY ESTRUTURADO (ideal para scripts):")
    # ── Query em CSV sem cabeçalho e sem unidades ───────────────────────────
    !nvidia-smi --query-gpu=index,name,temperature.gpu,utilization.gpu,utilization.memory,memory.used,memory.total,power.draw,power.limit,clocks.current.graphics,clocks.current.memory --format=csv,noheader,nounits

    print("\n" + "="*70)
    print("PROCESSOS NA GPU:")
    # ── Processos que usam a GPU ────────────────────────────────────────────
    !nvidia-smi --query-compute-apps=pid,process_name,used_gpu_memory --format=csv,noheader
else:
    print("Modo simulado: as queries acima exigem uma GPU NVIDIA real.")


## 3. Demo: Script de Coleta de Métricas em CSV

Criamos o script `monitor_gpu.sh`, que:
1. Recebe **intervalo**, **arquivo de saída** e **duração** por parâmetro.
2. Escreve o **cabeçalho** do CSV.
3. Em loop, coleta `nvidia-smi` com **timestamp** e uma linha por GPU.
4. Salva tudo em **CSV**.

> 💡 `set -euo pipefail` deixa o script rigoroso: aborta em erro (`-e`), em variável indefinida (`-u`) e em falha dentro de pipes (`pipefail`).


In [ ]:
%%writefile monitor_gpu.sh
#!/usr/bin/env bash
# monitor_gpu.sh — coleta métricas de GPU e salva em CSV
# Uso: ./monitor_gpu.sh [intervalo_segundos] [arquivo_saida] [duracao_segundos]
set -euo pipefail

INTERVALO="${1:-5}"
SAIDA="${2:-gpu_log_$(date +%Y%m%d_%H%M%S).csv}"
DURACAO="${3:-3600}"
if [ "$INTERVALO" -gt 0 ]; then
    MAX_AMOSTRAS=$(( DURACAO / INTERVALO ))
else
    MAX_AMOSTRAS=720
fi

CABECALHO="timestamp,gpu_index,gpu_name,temp_c,util_gpu_pct,"
CABECALHO+="util_mem_pct,mem_used_mb,mem_total_mb,power_w,power_limit_w,"
CABECALHO+="clock_graphics_mhz,clock_mem_mhz"

echo "$CABECALHO" > "$SAIDA"
echo "Iniciando monitoramento -> $SAIDA"
echo "Intervalo: ${INTERVALO}s | Duracao: ${DURACAO}s | Amostras: ${MAX_AMOSTRAS}"

AMOSTRA=0
while [ "$AMOSTRA" -lt "$MAX_AMOSTRAS" ]; do
    TS=$(date +"%Y-%m-%d %H:%M:%S")
    DADOS=$(nvidia-smi \
        --query-gpu=index,name,temperature.gpu,utilization.gpu,\
utilization.memory,memory.used,memory.total,\
power.draw,power.limit,clocks.current.graphics,clocks.current.memory \
        --format=csv,noheader,nounits)

    while IFS= read -r linha; do
        echo "$TS,$linha" >> "$SAIDA"
    done <<< "$DADOS"

    AMOSTRA=$(( AMOSTRA + 1 ))
    echo -ne "  Amostra $AMOSTRA/$MAX_AMOSTRAS\r"
    sleep "$INTERVALO"
done

echo ""
echo "Coleta concluida. Arquivo: $SAIDA"


In [ ]:
# @title ▶️ Executar o monitoramento (curto, para a aula)
# ============================================================================
# OBJETIVO: Rodar o monitor_gpu.sh por poucos segundos e gerar o CSV.
# Em produção usaríamos duração de horas (ex.: 3600 s = 1 h).
# ============================================================================
import pandas as pd

ARQUIVO_CSV = "gpu_log.csv"

if TEM_GPU:
    # 2 s de intervalo, 20 amostras (~40 s de coleta real)
    !chmod +x monitor_gpu.sh
    !./monitor_gpu.sh 2 {ARQUIVO_CSV} 40
else:
    # ── MODO SIMULADO: gera dados sintéticos com o MESMO schema do CSV real ──
    import numpy as np
    from datetime import datetime, timedelta

    print("⚠️  Gerando CSV simulado (sem GPU real)...")
    n = 20
    inicio = datetime.now()
    registros = []
    for i in range(n):
        ts = (inicio + timedelta(seconds=2 * i)).strftime("%Y-%m-%d %H:%M:%S")
        temp = 62 + 22 * np.sin(i / 3.0) + np.random.normal(0, 1.5)  # ~40 a 86 C
        util = np.clip(70 + 25 * np.sin(i / 2.0) + np.random.normal(0, 5), 0, 100)
        registros.append({
            "timestamp": ts, "gpu_index": 0, "gpu_name": "Tesla T4 (sim)",
            "temp_c": round(temp, 1), "util_gpu_pct": round(util, 0),
            "util_mem_pct": round(util * 0.6, 0), "mem_used_mb": 8000,
            "mem_total_mb": 15360, "power_w": round(45 + util * 0.2, 1),
            "power_limit_w": 70.0, "clock_graphics_mhz": 1590, "clock_mem_mhz": 5001,
        })
    pd.DataFrame(registros).to_csv(ARQUIVO_CSV, index=False)
    print(f"CSV simulado criado: {ARQUIVO_CSV}")

# ── Pré-visualização ────────────────────────────────────────────────────────
df = pd.read_csv(ARQUIVO_CSV)
print(f"\n📄 {len(df)} amostras em {ARQUIVO_CSV}")
df.head()


## 4. Teoria: Agendamento com `cron` e `systemd timers`

Para monitoramento **24h/7d** os scripts precisam ser agendados.

### Sintaxe do cron
```
┌────── minuto    (0–59)
│ ┌──── hora      (0–23)
│ │ ┌── dia/mês   (1–31)
│ │ │ ┌─ mês      (1–12)
│ │ │ │ ┌ dia/semana (0–7, 0 e 7 = domingo)
│ │ │ │ │
* * * * * comando
```

### Exemplos práticos
```bash
# Coletar métricas a cada 5 min no horário comercial (seg-sex)
*/5 8-20 * * 1-5 /home/user/scripts/monitor_gpu.sh 5 /data/logs/gpu.csv 300

# Verificar alertas a cada minuto
* * * * * /home/user/scripts/alerta_gpu.sh 80 95

# Relatório diário às 23:59
59 23 * * * /home/user/scripts/gerar_relatorio.sh

# Compactar logs com mais de 7 dias (domingos 00:01)
1 0 * * 0 find /data/logs/ -name "gpu_*.csv" -mtime +7 -exec gzip {} \;
```

### Comparativo

| Característica | cron | systemd timer |
| :--- | :--- | :--- |
| Configuração | `crontab -e` | arquivo `.timer` |
| Log automático | ❌ manual | ✅ journald |
| Dependências | ❌ não suporta | ✅ `After=` |
| Execução após boot | ❌ não | ✅ `Persistent=true` |
| Monitoramento | ❌ básico | ✅ `systemctl status` |
| Complexidade setup | ⭐ baixa | ⭐⭐ média |

> ⚠️ **No Colab não há cron nem systemd** (runtime efêmero, sem init). Demonstramos a sintaxe e simulamos o agendamento com um loop Python. Em servidores Linux reais, usamos o crontab.


In [ ]:
# @title ⏰ Simulação de agendamento (equivalente ao cron no Colab)
# ============================================================================
# OBJETIVO: Como não há cron no Colab, simulamos "coletar a cada N s por M ciclos".
# Em servidor real, isso seria uma linha do crontab (veja o markdown acima).
# ============================================================================
import time
from datetime import datetime

INTERVALO_CICLOS = 2   # segundos entre verificações (na aula)
N_CICLOS = 3           # repetições para não travar a aula

print("Simulando cron: verificação de alertas a cada", INTERVALO_CICLOS, "s")
for ciclo in range(1, N_CICLOS + 1):
    agora = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"  [{agora}] execução #{ciclo}: ./alerta_gpu.sh 80 95")
    time.sleep(INTERVALO_CICLOS)
print("✅ Simulação concluída. (Em produção: linha '* * * * * /caminho/alerta_gpu.sh 80 95')")


## 5. Alertas de Temperatura e Utilização (`alerta_gpu.sh`)

O script verifica **limites** e notifica por **Slack/Discord (webhook)** e **e-mail** (se `mail` existir).


In [ ]:
# @title 🚨 Executar alerta_gpu.sh (funciona com GPU real ou em modo simulado)
# ============================================================================
# OBJETIVO: Detectar GPUs acima dos limites e disparar alerta.
# Sem GPU, criamos um "nvidia-smi" falso a partir do CSV simulado, de modo que
# o MESMO script alerta_gpu.sh rode no Colab e dispare um alerta de exemplo.
# ============================================================================
import os
import subprocess
import pandas as pd

LOG = "/tmp/gpu_alertas.log"

# Limpa o log anterior para a demonstração ficar clara
if os.path.exists(LOG):
    os.remove(LOG)

if TEM_GPU:
    # Ambiente real: executa o script contra a GPU da máquina
    subprocess.run(["chmod", "+x", "alerta_gpu.sh"], check=False)
    subprocess.run(["bash", "alerta_gpu.sh", "80", "95"], check=False)
else:
    print("Modo simulado: criando um nvidia-smi falso a partir de gpu_log.csv...")
    df = pd.read_csv("gpu_log.csv")
    # Usa a amostra mais quente para garantir que o alerta dispare
    quente = df.loc[df["temp_c"].idxmax()]
    linha = (f"{int(quente['gpu_index'])}, {quente['gpu_name']}, "
             f"{int(round(quente['temp_c']))}, {int(round(quente['util_gpu_pct']))}")

    # Script "nvidia-smi" simulado que devolve o formato CSV esperado
    with open("nvidia-smi", "w", encoding="utf-8") as f:
        f.write(
            "#!/usr/bin/env bash\n"
            'echo "nvidia-smi simulado (Colab sem GPU)" >&2\n'
            f'echo "{linha}"\n'
        )
    os.chmod("nvidia-smi", 0o755)

    # Executa o alerta_gpu.sh real, com "." na frente do PATH
    ambiente = dict(os.environ, PATH=".:" + os.environ["PATH"])
    subprocess.run(["bash", "alerta_gpu.sh", "80", "95"], env=ambiente, check=False)

print("\n📄 Log de alertas:")
if os.path.exists(LOG):
    print(open(LOG, encoding="utf-8").read())
else:
    print("(nenhum alerta disparado)")


## 6. Atividade: Dashboard com `gnuplot` ou Matplotlib

Geramos um **dashboard de 4 painéis**:
1. 🌡️ **Temperatura** (com linha de limite em 80 °C)
2. ⚙️ **Utilização** (GPU e memória)
3. 💾 **Memória VRAM**
4. ⚡ **Potência**

No Colab usamos **Matplotlib** (já instalado e mais confiável). A versão **gnuplot** está no arquivo `gerar_graficos.sh` para uso em servidores Linux.


In [ ]:
# @title (Opcional) Instalar gnuplot no Colab
# ============================================================================
# OBJETIVO: Mostrar que o gnuplot é instalável no Colab via apt-get.
# Em produção, o script gerar_graficos.sh usa exatamente este utilitário.
# ============================================================================
!apt-get -qq install -y gnuplot > /dev/null 2>&1 && echo "✅ gnuplot instalado" || echo "⚠️  Falha ao instalar gnuplot"


In [ ]:
# @title 📊 Dashboard de 4 gráficos com Matplotlib (a partir do CSV)
# ============================================================================
# OBJETIVO: Visualizar temperatura, utilização, VRAM e potência ao longo do tempo.
# Lemos o CSV gerado (real ou simulado) e plotamos os 4 painéis.
# ============================================================================
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("gpu_log.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])  # converte texto -> datetime

fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle("GPU Monitoring Dashboard — Aula 14", fontsize=15, fontweight="bold")

# ── Painel 1: Temperatura ───────────────────────────────────────────────────
ax = axes[0, 0]
ax.plot(df["timestamp"], df["temp_c"], lw=2, color="#EF4444", label="Temperatura")
ax.axhline(80, color="#F97316", ls="--", lw=1, label="Limite 80 C")
ax.set_title("Temperatura GPU")
ax.set_ylabel("Temperatura (C)")
ax.set_ylim(0, 100)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# ── Painel 2: Utilização ────────────────────────────────────────────────────
ax = axes[0, 1]
ax.plot(df["timestamp"], df["util_gpu_pct"], lw=2, color="#10B981", label="GPU util")
ax.plot(df["timestamp"], df["util_mem_pct"], lw=2, color="#6366F1", label="Mem util")
ax.set_title("Utilizacao GPU")
ax.set_ylabel("Utilizacao (%)")
ax.set_ylim(0, 110)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# ── Painel 3: Memória VRAM ──────────────────────────────────────────────────
ax = axes[1, 0]
ax.fill_between(df["timestamp"], df["mem_used_mb"], color="#7E22CE", alpha=0.3)
ax.plot(df["timestamp"], df["mem_used_mb"], lw=2, color="#7E22CE", label="VRAM used")
ax.set_title("Memoria VRAM")
ax.set_ylabel("Memoria (MB)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# ── Painel 4: Potência ──────────────────────────────────────────────────────
ax = axes[1, 1]
ax.plot(df["timestamp"], df["power_w"], lw=2, color="#F97316", label="Power draw")
if "power_limit_w" in df.columns:
    ax.plot(df["timestamp"], df["power_limit_w"], lw=1, ls="--", color="#EF4444", label="Power limit")
ax.set_title("Potencia")
ax.set_ylabel("Potencia (W)")
ax.legend(loc="upper right")
ax.grid(alpha=0.3)

# Rotaciona os rótulos de tempo do eixo X para não sobreporem
for a in axes.flat:
    a.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig("gpu_dashboard.png", dpi=150, bbox_inches="tight")
plt.show()
print("📸 Dashboard salvo como 'gpu_dashboard.png'")


## 7. Integração com o Google Sheets

Publicar as métricas no Sheets permite acompanhar as GPUs de qualquer dispositivo, **sem SSH**.

### Configuração (fora do notebook)
1. Criar um projeto no **Google Cloud**.
2. Habilitar a **Google Sheets API**.
3. Criar uma **Service Account** e baixar o JSON.
4. **Compartilhar a planilha** com o e-mail da service account (permissão de editor).

### Segurança
- Credenciais via **variável de ambiente** (`SHEETS_ID`, `GOOGLE_CREDS`).
- JSON **fora do repositório** (nunca commitar).
- Service Account com **permissão mínima**.
- **Rotação de chaves** mensal.

> 🔒 A célula abaixo é um **template**: só executa se `SHEETS_ID` e o JSON existirem. Sem credenciais, ela apenas valida o CSV.


In [ ]:
# @title 📤 Publicar no Google Sheets (template — requer credenciais)
# ============================================================================
# OBJETIVO: Enviar as linhas do CSV para uma planilha via API.
# Só roda se SHEETS_ID e service_account.json estiverem presentes.
# ============================================================================
import csv, os

SHEETS_ID = os.environ.get("SHEETS_ID", "")
CRED_FILE = os.environ.get("GOOGLE_CREDS", "service_account.json")
RANGE_NAME = "GPU_Logs!A:M"

tem_credenciais = bool(SHEETS_ID) and os.path.exists(CRED_FILE)

if not tem_credenciais:
    print("🔒 SEM credenciais — modo validação.")
    print("   Defina SHEETS_ID e envie service_account.json para publicar de verdade.")
    with open("gpu_log.csv", encoding="utf-8") as f:
        reader = csv.reader(f)
        cabecalho = next(reader)
        linhas = list(reader)
    print(f"\nCabeçalho ({len(cabecalho)} colunas): {cabecalho}")
    print(f"Total de linhas prontas para envio: {len(linhas)}")
    print("\nPrimeira linha:", linhas[0] if linhas else "(vazio)")
else:
    from google.oauth2 import service_account
    from googleapiclient.discovery import build

    creds = service_account.Credentials.from_service_account_file(
        CRED_FILE, scopes=["https://www.googleapis.com/auth/spreadsheets"])
    service = build("sheets", "v4", credentials=creds)
    sheet = service.spreadsheets()

    with open("gpu_log.csv", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        linhas = list(reader)

    if linhas:
        resultado = sheet.values().append(
            spreadsheetId=SHEETS_ID, range=RANGE_NAME,
            valueInputOption="USER_ENTERED", body={"values": linhas}).execute()
        print(f"✅ Enviadas {resultado['updates']['updatedRows']} linhas ao Google Sheets.")
    else:
        print("Nenhum dado para enviar.")


## 8. Discussão em Grupo

Em grupos de 3–4, debatam com base no cenário do servidor de treinamento:

1. O script coleta a cada 5 s **no mesmo servidor do job**. Se o disco encher, o monitoramento para. Como tornar o sistema mais **robusto e independente** do job de treinamento?
2. `cron` e `systemd` são soluções de **servidor único**. Para um cluster com **50 nós GPU**, qual seria a arquitetura? (pesquise **Prometheus**, **Grafana** e **DCGM**.)
3. O alerta está em 80 °C, mas a GPU aguenta 95 °C. Qual o **limiar ideal**? Quais outros indicadores (`fan.speed`, *power throttle*, ECC) deveriam acionar alertas?
4. O Google Sheets tem **quota e latência**. Para produção real, quais ferramentas o substituiriam? Compare com **InfluxDB + Grafana**.


## 9. Síntese e Tarefa de Casa

### 🔑 Pontos-chave
- **`nvidia-smi --query-gpu`**: extrai métricas estruturadas em CSV — base de todo monitoramento.
- **`monitor_gpu.sh`**: loop Bash com timestamp e suporte a múltiplas GPUs.
- **`alerta_gpu.sh`**: verifica limites de temperatura/utilização e notifica via Slack/e-mail.
- **`cron`**: agendador clássico do Linux, simples para scripts periódicos.
- **`systemd timer`**: alternativa moderna com logging via journald e dependências.
- **`gnuplot`**: séries temporais direto no terminal, sem Python.

### 📌 Tarefa de Casa
Configure o sistema de monitoramento completo em um servidor com GPU (pode usar Colab + simulação):
1. Adapte `monitor_gpu.sh` para coletar **10 minutos** com intervalo de **3 s**.
2. Crie o dashboard (**gnuplot ou Python/Matplotlib**) com os **4 gráficos** da aula.
3. Configure um **cron job** que execute o monitoramento **todo dia às 08h**.
4. **Bônus:** implemente o alerta de temperatura com envio para **webhook Slack ou Discord**.

### 🛠️ Recursos
- [nvidia-smi docs](https://developer.nvidia.com/nvidia-system-management-interface)
- [GNU cron manual](https://www.gnu.org/software/coreutils/manual/html_node/crontab-invocation.html)
- [gnuplot](http://www.gnuplot.info/documentation.html)
- [Google Sheets API — Python Quickstart](https://developers.google.com/sheets/api/quickstart/python)
